# Fake News Detector

> **Objective:** Build a machine-learning pipeline that classifies a news article as **FAKE** or **REAL** based on its text content.

## Approach
1. Load and validate a labelled news dataset.
2. Convert raw text to TF-IDF features.
3. Train a **Passive Aggressive Classifier** — a fast, online linear model well-suited for text classification.
4. Evaluate with accuracy, precision, recall, F1-score, and a confusion matrix.
5. Provide a simple inference interface so anyone can test the model on their own text.
6. Persist the trained pipeline with `joblib` for later reuse.

---

## Requirements

| Package | Version (tested) | Purpose |
|---------|------------------|---------|
| Python  | ≥ 3.8            | Runtime |
| pandas  | ≥ 1.3            | Data loading & manipulation |
| numpy   | ≥ 1.21           | Numerical operations |
| scikit-learn | ≥ 1.0       | ML pipeline, vectorizer, classifier, metrics |
| matplotlib | ≥ 3.4         | Confusion matrix visualisation |
| joblib  | ≥ 1.0            | Model serialisation |

Install with:
```bash
pip install pandas numpy scikit-learn matplotlib joblib
```

**Dataset:** `news_datasets.csv` — a labelled CSV with at least `text` and `label` columns (values `FAKE` / `REAL`).
Place the file at `data/news_datasets.csv` relative to this notebook, or update `DATA_PATH` in the *Configuration* cell below.

---

## 1 · Setup & Imports

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings('ignore')
print('All imports successful.')

## 2 · Configuration

Adjust the variables below to change paths or model hyper-parameters.

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Data path ────────────────────────────────────────────────────────────────
# Primary: repo-relative path (works locally and in CI).
# Fallback: Kaggle input directory (works on Kaggle kernels).
DATA_PATH_PRIMARY = os.path.join(
    os.getcwd(), 'data', 'news_datasets.csv'
)
DATA_PATH_FALLBACK = (
    '../input/news-data-set-fake-news-with-python/news_datasets.csv'
)

# ── Train / test split ───────────────────────────────────────────────────────
TEST_SIZE = 0.30

# ── TF-IDF hyper-parameters ──────────────────────────────────────────────────
TFIDF_MAX_DF    = 0.7
TFIDF_STOP_WORDS = 'english'

# ── Classifier hyper-parameters ──────────────────────────────────────────────
PAC_MAX_ITER = 50

# ── Model persistence ────────────────────────────────────────────────────────
MODEL_DIR  = 'models'
MODEL_PATH = os.path.join(MODEL_DIR, 'fake_news_pipeline.joblib')
os.makedirs(MODEL_DIR, exist_ok=True)

print('Configuration loaded.')

## 3 · Data Loading

In [ ]:
REQUIRED_COLUMNS = {'text', 'label'}

def load_data(primary: str, fallback: str) -> pd.DataFrame:
    """Load the dataset from *primary* path; fall back to *fallback* if needed."""
    for path in (primary, fallback):
        if os.path.exists(path):
            print(f'Loading data from: {path}')
            df = pd.read_csv(path)
            missing = REQUIRED_COLUMNS - set(df.columns)
            if missing:
                raise ValueError(
                    f'Dataset is missing required column(s): {missing}. '
                    f'Found columns: {list(df.columns)}'
                )
            return df
    raise FileNotFoundError(
        'Dataset not found.\n'
        f'  Tried primary  : {primary}\n'
        f'  Tried fallback : {fallback}\n'
        'Please place news_datasets.csv in the data/ folder or update DATA_PATH_PRIMARY.'
    )

df = load_data(DATA_PATH_PRIMARY, DATA_PATH_FALLBACK)
print(f'Dataset loaded — {df.shape[0]:,} rows, {df.shape[1]} columns.')

## 4 · Exploratory Data Analysis

In [ ]:
# First five rows
df.head()

In [ ]:
# Shape and basic info
print('Shape :', df.shape)
print()
print('Null values per column:')
print(df.isnull().sum())

In [ ]:
# Label distribution
label_counts = df['label'].value_counts()
print('Label distribution:')
print(label_counts.to_string())
balance = label_counts.min() / label_counts.max()
print(f'\nClass balance: {balance:.2%} (1.0 = perfectly balanced)')

## 5 · Preprocessing & Train / Test Split

In [ ]:
# Drop rows where text or label is missing
df.dropna(subset=['text', 'label'], inplace=True)

X = df['text']
y = df['label']

# Stratified split — preserves class ratio in both subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')

## 6 · Build Pipeline

Wrapping the TF-IDF vectoriser and classifier in a single `sklearn.pipeline.Pipeline` ensures that:
- The vectoriser is **fit only on training data** (preventing data leakage).
- The whole pipeline can be serialised and loaded as one object.
- Hyper-parameter search (e.g. `GridSearchCV`) can be applied end-to-end.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=TFIDF_STOP_WORDS,
        max_df=TFIDF_MAX_DF,
    )),
    ('clf', PassiveAggressiveClassifier(
        max_iter=PAC_MAX_ITER,
        random_state=RANDOM_STATE,
    )),
])

print(pipeline)

## 7 · Training

In [ ]:
pipeline.fit(X_train, y_train)
print('Training complete.')

## 8 · Evaluation

In [ ]:
y_pred = pipeline.predict(X_test)

# ── Accuracy ─────────────────────────────────────────────────────────────────
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy : {accuracy * 100:.2f}%')
print()

# ── Precision / Recall / F1 ──────────────────────────────────────────────────
print('Classification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
labels_order = sorted(y.unique())  # consistent ordering: FAKE, REAL

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=labels_order,
    cmap='Blues',
    ax=ax,
)
ax.set_title('Confusion Matrix — Fake News Detector')
plt.tight_layout()
plt.show()

## 9 · Inference

Paste any news snippet into `news_snippet` below and run the cell to get a prediction.

In [ ]:
def predict_news(text: str, model: Pipeline) -> str:
    """Return 'FAKE' or 'REAL' for the given news text."""
    prediction = model.predict([text])[0]
    return prediction


# ── Demo ─────────────────────────────────────────────────────────────────────
news_snippet = (
    "Scientists confirm that regular exercise significantly reduces the risk "
    "of cardiovascular disease, according to a new study published in the "
    "New England Journal of Medicine."
)

result = predict_news(news_snippet, pipeline)
print(f'Prediction: {result}')

## 10 · Save & Load the Model

The trained pipeline is saved to `models/fake_news_pipeline.joblib`.

In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────────
joblib.dump(pipeline, MODEL_PATH)
print(f'Pipeline saved to: {MODEL_PATH}')

# ── Load & verify ────────────────────────────────────────────────────────────
loaded_pipeline = joblib.load(MODEL_PATH)
sample_pred = predict_news(news_snippet, loaded_pipeline)
print(f'Loaded pipeline prediction: {sample_pred}')
print('Save / load verified successfully.')